In [ ]:
import pandas as pd
import numpy as np
import statistics as stat
import tabula as tab
from pathlib import Path

from tempfile import gettempdir

## Import and clean both PDF pages

Consumption is supplied food in kg per person per year; CO2 columns are kg CO2 per person per year. The merged PDF headers are replaced with unique names. `df` contains one row per country.

In [ ]:
# Locate the PDF whether the notebook runs from practice/ or the project root.
pdf_path = next(
    folder / "data" / "Food-Carbon-Footprint-Index-2018.pdf"
    for parent in [Path.cwd(), *Path.cwd().parents]
    for folder in [parent, parent / "Data Science"]
    if (folder / "data" / "Food-Carbon-Footprint-Index-2018.pdf").is_file()
)

# Read BOTH pages. Keep the PDF headers as ordinary rows for now.
tables = tab.read_pdf(
    str(pdf_path),
    pages="1-2",
    lattice=True,
    force_subprocess=True,
    java_options=["-Dfile.encoding=UTF-8", "-Dsun.stderr.encoding=UTF-8",
                  f"-Dpdfbox.fontcache={gettempdir()}"],
    pandas_options={"header": None},
)

animal_foods = ["pork", "poultry", "beef", "lamb_goat", "fish", "eggs", "milk_cheese"]
plant_foods = ["wheat", "rice", "soybeans", "nuts_peanut_butter"]

def food_columns(foods):
    return [f"{food}_{measure}_kg_per_person_year"
            for food in foods for measure in ["consumption", "co2"]]

columns = (
    ["rank", "country"]
    + food_columns(animal_foods)
    + ["animal_total_co2_kg_per_person_year"]
    + food_columns(plant_foods)
    + ["plant_total_co2_kg_per_person_year",
       "animal_minus_plant_co2_kg_per_person_year"]
)

# A numeric rank identifies a country row, removing headers without losing
# the first countries on page 2 (which has no repeated header).
frames = []
for table in tables:
    if table.shape[1] != len(columns):
        raise ValueError(f"Expected {len(columns)} columns, got {table.shape[1]}")
    country_rows = pd.to_numeric(table.iloc[:, 0], errors="coerce").notna()
    frames.append(table.loc[country_rows].copy())

df = pd.concat(frames, ignore_index=True)
df.columns = columns
df["country"] = df["country"].str.strip()
for column in df.columns.drop("country"):
    df[column] = pd.to_numeric(df[column], errors="raise")
df["rank"] = df["rank"].astype(int)

print(f"Loaded {len(df)} countries from both pages, with {len(df.columns)} columns.")
display(df.head())
